<div align="center">

# **Tesis: Conectividad Estructural de Tractos Témporo-Límbicos en Epilepsia**
### Archivo 3 de 3 — Modelado: CNN sobre Imágenes TDI y Benchmarking de Machine Learning Clásico

</div>

Este notebook es la **siguiente etapa** del pipeline, y consume directamente los artefactos que
generan los dos notebooks anteriores:

- **Del Archivo 1** (`features_por_sujeto.csv`): la matriz tabular de métricas escalares y
  geométricas, una fila por sujeto.
- **Del Archivo 2** (`tdi_grises/*_gris.png`): los cortes TDI en escala de grises normalizada, por
  clúster y por sujeto (Sección 6 del Archivo 2).

Implementa **tres enfoques de clasificación independientes** (epilepsia vs. control), para
comparar su desempeño:

1. **Sección A — Modelo A, CNN sobre imágenes TDI:** aprende patrones discriminativos directamente
   de la geometría espacial 2D de los tractos (imagen únicamente), sin reducirla primero a un
   puñado de métricas escalares.
2. **Sección A2 — Modelo B, fusión CNN + tabular:** la misma imagen TDI, combinada con las métricas
   reales de difusión (FA/MD/AD/RD) de ese mismo tracto que exporta el Archivo 2 (Sección 3.5), en
   una red de dos ramas — para medir si agregar la microestructura mejora sobre la imagen sola.
3. **Sección B — Benchmarking de 5 clasificadores clásicos** sobre las métricas tabulares ya
   extraídas (FA, MD, AD, RD, volumen, geometría), con selección recursiva de características
   (RFECV).

> ⚠️ **Corrección respecto a un borrador previo de este notebook:** una versión anterior entrenaba
> los modelos con **etiquetas generadas aleatoriamente** (`np.random.randint` o reglas arbitrarias
> sobre el nombre del sujeto, sin relación con un diagnóstico real) — el *accuracy* resultante de
> eso no mide nada clínicamente válido, por más alto que salga el número. Esta versión **siempre**
> deriva `grupo_dx` de metadatos reales (nombre de archivo/carpeta) y, solo cuando no hay datos
> reales disponibles, genera un conjunto sintético **explícitamente etiquetado como tal**, con una
> señal estadística conocida y controlada (no ruido puro), únicamente para verificar que el pipeline
> corre de principio a fin.


# **0. Configuración del entorno**

In [ ]:
import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", font_scale=0.85)
pd.set_option("display.max_colwidth", 120)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CARPETA_SALIDA = "/content/" if IN_COLAB else "./"
os.makedirs(CARPETA_SALIDA, exist_ok=True)

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local / Jupyter'}")
print(f"Carpeta de salida: {CARPETA_SALIDA}")


## **0.1. Ubicación de los artefactos de los Archivos 1 y 2**
Ajusta estas dos rutas a donde tengas, respectivamente, el `features_por_sujeto.csv` exportado por
el Archivo 1, y la carpeta raíz que contiene una subcarpeta por sujeto con sus `tdi_grises/*.png`
exportados por el Archivo 2 (ver la convención de nombre de subcarpeta en la Sección 1.2).

In [ ]:
RUTA_FEATURES_TABULARES = CARPETA_SALIDA + "features_por_sujeto.csv"
CARPETA_TDI_COHORTE = CARPETA_SALIDA + "tdi_cohorte/"
os.makedirs(CARPETA_TDI_COHORTE, exist_ok=True)

if IN_COLAB:
    print("Si ya tenés 'features_por_sujeto.csv' (Archivo 1) y/o la carpeta de TDI por sujeto "
          "(Archivo 2), subilos ahora a este entorno (p. ej. arrastrándolos al panel de archivos "
          "de la izquierda, o montando Google Drive) y ajustá las dos rutas de arriba si hace "
          "falta. Si no los tenés todavía, el notebook sigue funcionando con datos sintéticos "
          "para que puedas validar que el pipeline corre de punta a punta.")

print(f"\nRuta features tabulares: {RUTA_FEATURES_TABULARES}  "
      f"(existe: {os.path.exists(RUTA_FEATURES_TABULARES)})")
print(f"Carpeta TDI por cohorte: {CARPETA_TDI_COHORTE}  "
      f"(subcarpetas encontradas: {len(glob.glob(CARPETA_TDI_COHORTE + '*/'))})")


# **Sección A — CNN sobre Imágenes TDI**

## **A.1. Formulación del enfoque visual**
Las métricas escalares (FA, MD, volumen) resumen cada tracto en un puñado de números, perdiendo su
forma espacial completa. Para aprovechar esa información geométrica, se entrena una **Red Neuronal
Convolucional (CNN)** directamente sobre los cortes TDI en escala de grises que exporta el Archivo 2
(Sección 6.2) — una imagen 2D por clúster y por corte anatómico, que conserva la morfología real del
tracto (curvatura, elongación, grosor) en vez de reducirla a un promedio.

## **A.2. Unidad de análisis: corte por clúster, no por sujeto**
Cada sujeto aporta varias imágenes (hasta 4 tractos × 2 lados = 8 clústeres, un corte axial cada
uno). Se entrena la CNN a **nivel de clúster** (cada imagen es una muestra independiente, etiquetada
con el `grupo_dx` de su sujeto), lo cual es una práctica habitual en imagenología médica con N de
pacientes reducido (más muestras de entrenamiento que sujetos). La partición train/test se hace,
igual que en el Archivo 1, **agrupada por `subject_id`** (`GroupShuffleSplit`) — para que ninguna
imagen de un sujeto del conjunto de prueba haya sido vista, aunque sea de otro tracto, durante el
entrenamiento.

## **A.3. Convención de carpetas esperada**
```
tdi_cohorte/
├── S1_epi_corrected/
│   └── tdi_grises/
│       ├── cluster_00_derecho_axial_gris.png
│       ├── cluster_01_izquierdo_axial_gris.png
│       └── ...
├── S2_control_corrected/
│   └── tdi_grises/
│       └── ...
```
Es decir: una subcarpeta por sujeto (el nombre debe contener el patrón de grupo —`_epi_`,
`epi_corrected`, `_tle_`, `_elt_` para epilepsia, o `_control_`, `_hc_`, `control_corrected` para
control—, igual que en el Archivo 1), cada una con su propia carpeta `tdi_grises/` tal como la
genera el Archivo 2 al correrlo para ese sujeto.

In [ ]:
PATRON_GRUPO_EPILEPSIA = re.compile(r"(?:_epi_|epi_corrected|_epilepsy_|_tle_|_elt_)")
PATRON_GRUPO_CONTROL = re.compile(r"(?:_control_|_hc_|control_corrected)")
PATRON_SUBJECT_ID = re.compile(r"(s\d+)")


def inferir_metadatos_carpeta_sujeto(nombre_carpeta):
    """Infiere subject_id y grupo_dx a partir del nombre de la subcarpeta del
    sujeto (misma convención de patrones que el Archivo 1, Sección 2.3)."""
    n = nombre_carpeta.lower()
    m_id = PATRON_SUBJECT_ID.search(n)
    subject_id = m_id.group(1).upper() if m_id else nombre_carpeta

    if PATRON_GRUPO_EPILEPSIA.search(n):
        grupo_dx = "epilepsia"
    elif PATRON_GRUPO_CONTROL.search(n):
        grupo_dx = "control"
    else:
        grupo_dx = "desconocido"
    return subject_id, grupo_dx


def listar_imagenes_tdi(carpeta_cohorte, corte="axial"):
    """Recorre 'carpeta_cohorte' y arma una lista de dicts {ruta, subject_id,
    grupo_dx, cluster} para cada imagen '*_<corte>_gris.png' encontrada bajo
    cualquier subcarpeta 'tdi_grises/' de cada sujeto.
    """
    registros = []
    for carpeta_sujeto in sorted(glob.glob(os.path.join(carpeta_cohorte, "*/"))):
        nombre_sujeto = os.path.basename(os.path.normpath(carpeta_sujeto))
        subject_id, grupo_dx = inferir_metadatos_carpeta_sujeto(nombre_sujeto)

        patron_busqueda = os.path.join(carpeta_sujeto, "tdi_grises", f"*_{corte}_gris.png")
        for ruta_img in sorted(glob.glob(patron_busqueda)):
            cluster_match = re.search(r"(cluster_\d+_\w+)_" + corte, os.path.basename(ruta_img))
            registros.append({
                "ruta": ruta_img,
                "subject_id": subject_id,
                "grupo_dx": grupo_dx,
                "cluster": cluster_match.group(1) if cluster_match else "desconocido",
            })
    return pd.DataFrame(registros)


df_imagenes = listar_imagenes_tdi(CARPETA_TDI_COHORTE, corte="axial")
print(f"Imágenes TDI (corte axial) encontradas: {len(df_imagenes)}")
if len(df_imagenes):
    print(f"Sujetos: {df_imagenes['subject_id'].nunique()}  "
          f"(grupo_dx: {df_imagenes['grupo_dx'].value_counts().to_dict()})")
df_imagenes.head()


## **A.4. Carga de imágenes (o generación de un set sintético de validación)**
Si no se encontraron imágenes reales (Sección A.3 vacía), se genera un set sintético **con señal
controlada y claramente marcado como tal** — nunca etiquetas aleatorias desconectadas de los datos —
únicamente para verificar que el resto del pipeline (carga → CNN → evaluación) corre sin errores.

In [ ]:
from PIL import Image

TAMANO_IMG = 128  # input_shape de la CNN (Sección A.5): (TAMANO_IMG, TAMANO_IMG, 1)


def cargar_y_redimensionar(ruta, tamano=TAMANO_IMG):
    """Carga una imagen en escala de grises, la redimensiona a (tamano, tamano)
    y normaliza su intensidad a [0, 1].
    """
    img = Image.open(ruta).convert("L").resize((tamano, tamano))
    return np.asarray(img, dtype=np.float32) / 255.0


def generar_set_sintetico_imagenes(n_sujetos=40, n_clusters_por_sujeto=4, tamano=TAMANO_IMG,
                                    semilla=42):
    """Genera imágenes sintéticas con una señal controlada y conocida (un
    gradiente diagonal cuya pendiente difiere, en promedio, entre 'epilepsia'
    y 'control', más ruido gaussiano) -- NUNCA etiquetas desconectadas de los
    datos. Sirve solo para validar que el pipeline de carga/entrenamiento/
    evaluación corre de punta a punta antes de tener imágenes reales.
    """
    rng = np.random.default_rng(semilla)
    xs, ys, subject_ids, grupos = [], [], [], []
    coord = np.linspace(0, 1, tamano)
    malla_x, malla_y = np.meshgrid(coord, coord)

    for i in range(n_sujetos):
        grupo = "epilepsia" if i % 2 == 0 else "control"
        pendiente = 1.6 if grupo == "epilepsia" else 1.0
        for c in range(n_clusters_por_sujeto):
            patron = np.sin(pendiente * (malla_x + malla_y) * np.pi) * 0.4 + 0.5
            ruido = rng.normal(0, 0.05, size=(tamano, tamano))
            xs.append(np.clip(patron + ruido, 0, 1).astype(np.float32))
            subject_ids.append(f"SIM{i:03d}")
            grupos.append(grupo)

    return np.array(xs), np.array(subject_ids), np.array(grupos)


if len(df_imagenes):
    X_img = np.stack([cargar_y_redimensionar(r) for r in df_imagenes["ruta"]])
    subject_ids_img = df_imagenes["subject_id"].values
    grupos_img = df_imagenes["grupo_dx"].values
    print(f"Imágenes reales cargadas: {X_img.shape}")
else:
    print("No se encontraron imágenes reales en CARPETA_TDI_COHORTE; generando set SINTÉTICO "
          "de validación (ver nota metodológica más arriba)...")
    X_img, subject_ids_img, grupos_img = generar_set_sintetico_imagenes()
    print(f"Imágenes sintéticas generadas: {X_img.shape}")

X_img = X_img[..., np.newaxis]  # (N, TAMANO_IMG, TAMANO_IMG, 1) -- canal único (escala de grises)
y_img = (grupos_img == "epilepsia").astype(int)
print(f"Forma final de X_img: {X_img.shape}   Distribución de clases: "
      f"control={np.sum(y_img == 0)}, epilepsia={np.sum(y_img == 1)}")


## **A.5. Arquitectura de la CNN**
Tres bloques convolucionales (filtros 3×3, `BatchNormalization`, `MaxPooling2D`) seguidos de una
capa densa con `Dropout` (40%) y una salida sigmoide para clasificación binaria.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models


def construir_modelo_cnn(input_shape=(TAMANO_IMG, TAMANO_IMG, 1)):
    """Construye una CNN para clasificar cortes TDI (epilepsia vs. control)."""
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


modelo_cnn = construir_modelo_cnn()
modelo_cnn.summary()


## **A.6. Partición agrupada por sujeto, entrenamiento y evaluación**

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              ConfusionMatrixDisplay)

splitter_img = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
idx_train_img, idx_test_img = next(splitter_img.split(X_img, y_img, groups=subject_ids_img))

X_train_img, X_test_img = X_img[idx_train_img], X_img[idx_test_img]
y_train_img, y_test_img = y_img[idx_train_img], y_img[idx_test_img]

print(f"Entrenamiento: {len(X_train_img)} imágenes de {len(set(subject_ids_img[idx_train_img]))} "
      f"sujetos  |  Prueba: {len(X_test_img)} imágenes de "
      f"{len(set(subject_ids_img[idx_test_img]))} sujetos")

# Peso balanceado por clase (el nº de imágenes por clase puede no ser exactamente igual)
n_pos, n_neg = y_train_img.sum(), len(y_train_img) - y_train_img.sum()
peso_clases = {0: 1.0, 1: (n_neg / max(n_pos, 1))} if n_pos > 0 else {0: 1.0, 1: 1.0}

callback_parada_temprana = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True
)

historial = modelo_cnn.fit(
    X_train_img, y_train_img,
    epochs=60, batch_size=8, validation_split=0.2,
    class_weight=peso_clases, callbacks=[callback_parada_temprana], verbose=0,
)
print(f"Entrenamiento finalizado en {len(historial.history['loss'])} épocas "
      f"(parada temprana monitoreando val_loss).")


In [ ]:
plt.figure(figsize=(10, 3.5))
plt.subplot(1, 2, 1)
plt.plot(historial.history["loss"], label="Entrenamiento")
plt.plot(historial.history["val_loss"], label="Validación")
plt.title("Pérdida (binary cross-entropy)")
plt.xlabel("Época")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(historial.history["accuracy"], label="Entrenamiento")
plt.plot(historial.history["val_accuracy"], label="Validación")
plt.title("Accuracy")
plt.xlabel("Época")
plt.legend()

plt.tight_layout()
ruta_curvas_png = CARPETA_SALIDA + "cnn_curvas_entrenamiento.png"
plt.savefig(ruta_curvas_png, dpi=140, bbox_inches="tight")
plt.show()
print(f"Curvas guardadas en: {ruta_curvas_png}")


In [ ]:
y_proba_cnn = modelo_cnn.predict(X_test_img, verbose=0).ravel()
y_pred_cnn = (y_proba_cnn >= 0.5).astype(int)

print("=== CNN — Evaluación a nivel de imagen (conjunto de prueba, sujetos no vistos) ===")
print(classification_report(y_test_img, y_pred_cnn,
                             target_names=["Control (0)", "Epilepsia (1)"], zero_division=0))

if len(set(y_test_img)) > 1:
    auc_cnn = roc_auc_score(y_test_img, y_proba_cnn)
    print(f"AUC-ROC: {auc_cnn:.3f}")
else:
    auc_cnn = float("nan")
    print("AUC-ROC no definido: el conjunto de prueba solo tiene una clase.")

matriz_confusion_cnn = confusion_matrix(y_test_img, y_pred_cnn)
ConfusionMatrixDisplay(matriz_confusion_cnn, display_labels=["Control", "Epilepsia"]).plot(
    cmap="Blues", colorbar=False
)
plt.title("CNN — Matriz de confusión (nivel de imagen)")
ruta_matriz_cnn_png = CARPETA_SALIDA + "cnn_matriz_confusion.png"
plt.savefig(ruta_matriz_cnn_png, dpi=140, bbox_inches="tight")
plt.show()


## **A.7. Agregación a nivel de sujeto (voto mayoritario)**
La Sección A.6 evalúa a nivel de imagen (clúster). Para obtener una decisión clínica por **paciente**
—la unidad que realmente importa—, se agregan las predicciones de todas las imágenes de prueba de un
mismo sujeto por voto mayoritario de probabilidad promedio.

In [ ]:
df_pred_img = pd.DataFrame({
    "subject_id": subject_ids_img[idx_test_img],
    "y_real": y_test_img,
    "y_proba": y_proba_cnn,
})

df_pred_sujeto = df_pred_img.groupby("subject_id").agg(
    y_real=("y_real", "first"), y_proba_media=("y_proba", "mean")
).reset_index()
df_pred_sujeto["y_pred"] = (df_pred_sujeto["y_proba_media"] >= 0.5).astype(int)

print("=== CNN — Evaluación agregada a nivel de SUJETO (voto por promedio de probabilidad) ===")
print(classification_report(df_pred_sujeto["y_real"], df_pred_sujeto["y_pred"],
                             target_names=["Control (0)", "Epilepsia (1)"], zero_division=0))
df_pred_sujeto.round(3)


# **Sección A2 — Modelo B: Fusión de Imagen TDI + Métricas Tabulares (FA/MD/AD/RD)**

## **A2.1. Motivación**
El Modelo A (Sección A) solo "ve" la forma geométrica del tracto (TDI en escala de grises). Pero el
Archivo 2 (Sección 3.5) ya extrae, para ese mismo clúster, sus métricas de difusión reales —FA, MD,
AD, RD—, que capturan información microestructural que una imagen de densidad de streamlines no
contiene (dos tractos pueden tener una forma muy similar y una FA muy distinta). El **Modelo B** es
una red de **dos ramas** que combina ambas fuentes:

- **Rama de imagen:** los mismos tres bloques convolucionales del Modelo A, hasta un vector de
  características (*embedding*) de 64 dimensiones.
- **Rama tabular:** un perceptrón de 2 capas sobre las 9 métricas escalares de
  `metricas_reales_por_tracto.csv` (`fa_mean`, `fa_sd`, `md_mean`, `md_sd`, `ad_mean`, `ad_sd`,
  `rd_mean`, `rd_sd`, `n_streamlines`), estandarizadas.
- Ambos vectores se **concatenan** y pasan por una capa densa final antes de la salida sigmoide.

La unidad de análisis sigue siendo el clúster (Sección A.2): cada imagen se empareja **1 a 1**, por
`(subject_id, cluster_tag)`, con su fila de métricas tabulares del mismo tracto y mismo sujeto —
ambas provienen de la misma ejecución del Archivo 2, así que el emparejamiento es exacto.

## **A2.2. Carga y emparejamiento de las dos fuentes**
Se asume que cada subcarpeta de sujeto en `CARPETA_TDI_COHORTE` (Sección A.3) también contiene,
en su raíz, el `<subject_id>_metricas_reales_por_tracto.csv` que exporta el Archivo 2 (Sección 3.5).

In [ ]:
def cargar_metricas_reales_cohorte(carpeta_cohorte):
    """Busca, en cada subcarpeta de sujeto de 'carpeta_cohorte', un archivo
    '*_metricas_reales_por_tracto.csv' (exportado por el Archivo 2, Sección 3.5)
    y los concatena en un único DataFrame de la cohorte.
    """
    tablas = []
    for ruta_csv in sorted(glob.glob(os.path.join(carpeta_cohorte, "*", "*_metricas_reales_por_tracto.csv"))):
        tablas.append(pd.read_csv(ruta_csv))
    if not tablas:
        return pd.DataFrame()
    return pd.concat(tablas, ignore_index=True)


COLUMNAS_TABULARES_MODELO_B = ["fa_mean", "fa_sd", "md_mean", "md_sd",
                                "ad_mean", "ad_sd", "rd_mean", "rd_sd", "n_streamlines"]

df_metricas_cohorte = cargar_metricas_reales_cohorte(CARPETA_TDI_COHORTE)
print(f"Filas de métricas reales (Archivo 2, Sección 3.5) encontradas: {len(df_metricas_cohorte)}")

if len(df_imagenes) and len(df_metricas_cohorte):
    df_emparejado = df_imagenes.merge(
        df_metricas_cohorte, left_on=["subject_id", "cluster"],
        right_on=["subject_id", "cluster_tag"], how="inner",
    )
    print(f"Imágenes emparejadas con su fila tabular: {len(df_emparejado)} / {len(df_imagenes)}")
else:
    df_emparejado = pd.DataFrame()
    print("No hay suficientes datos reales emparejados todavía (se necesitan imágenes del "
          "Archivo 2, Sección 6, y metricas_reales_por_tracto.csv del Archivo 2, Sección 3.5, "
          "para los mismos sujetos). Se usará un set sintético PAREADO para validar el pipeline.")


## **A2.3. Set sintético pareado (solo si no hay datos reales emparejados)**
A diferencia de un set sintético ingenuo, aquí la señal tabular y la señal de imagen están
**correlacionadas entre sí y con el grupo** — simula la relación real que se espera entre
microestructura (FA/MD) y morfología (forma del TDI), no dos fuentes independientes de ruido.

In [ ]:
def generar_set_sintetico_pareado(n_sujetos=40, n_clusters_por_sujeto=4, tamano=TAMANO_IMG,
                                   semilla=42):
    rng = np.random.default_rng(semilla)
    imagenes, tabulares, subject_ids, grupos = [], [], [], []
    coord = np.linspace(0, 1, tamano)
    malla_x, malla_y = np.meshgrid(coord, coord)

    for i in range(n_sujetos):
        grupo = "epilepsia" if i % 2 == 0 else "control"
        pendiente = 1.6 if grupo == "epilepsia" else 1.0
        fa_base = 0.38 if grupo == "epilepsia" else 0.45   # FA menor en epilepsia (simulado)
        for c in range(n_clusters_por_sujeto):
            patron = np.sin(pendiente * (malla_x + malla_y) * np.pi) * 0.4 + 0.5
            imagenes.append(np.clip(patron + rng.normal(0, 0.05, (tamano, tamano)), 0, 1))

            tabulares.append([
                rng.normal(fa_base, 0.04),            # fa_mean
                rng.normal(0.05, 0.01),                # fa_sd
                rng.normal(0.0009 if grupo == "epilepsia" else 0.0008, 0.00005),  # md_mean
                rng.normal(0.0001, 0.00002),           # md_sd
                rng.normal(0.0012, 0.0001),            # ad_mean
                rng.normal(0.00015, 0.00003),          # ad_sd
                rng.normal(0.0007, 0.0001),            # rd_mean
                rng.normal(0.0001, 0.00002),           # rd_sd
                rng.normal(15000, 3000),               # n_streamlines
            ])
            subject_ids.append(f"SIM{i:03d}")
            grupos.append(grupo)

    return (np.array(imagenes, dtype=np.float32), np.array(tabulares, dtype=np.float32),
            np.array(subject_ids), np.array(grupos))


if len(df_emparejado):
    X_img_b = np.stack([cargar_y_redimensionar(r) for r in df_emparejado["ruta"]])
    X_tab_b = df_emparejado[COLUMNAS_TABULARES_MODELO_B].values.astype(np.float32)
    subject_ids_b = df_emparejado["subject_id"].values
    grupos_b = df_emparejado["grupo_dx_x"].values
else:
    X_img_b, X_tab_b, subject_ids_b, grupos_b = generar_set_sintetico_pareado()
    print(f"Set sintético pareado generado: {X_img_b.shape[0]} muestras.")

X_img_b = X_img_b[..., np.newaxis]
y_b = (grupos_b == "epilepsia").astype(int)
print(f"X_img_b: {X_img_b.shape}   X_tab_b: {X_tab_b.shape}   "
      f"clases: control={np.sum(y_b == 0)}, epilepsia={np.sum(y_b == 1)}")


## **A2.4. Estandarización de la rama tabular, partición y arquitectura de fusión**
La rama tabular se estandariza con `StandardScaler` (ajustado **solo** sobre el conjunto de
entrenamiento, para no filtrar información del conjunto de prueba) antes de entrar a la red.

In [ ]:
from tensorflow.keras import Model
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

splitter_b = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
idx_train_b, idx_test_b = next(splitter_b.split(X_img_b, y_b, groups=subject_ids_b))

escalador_tabular = StandardScaler()
X_tab_train_b = escalador_tabular.fit_transform(
    SimpleImputer(strategy="median").fit_transform(X_tab_b[idx_train_b])
)
X_tab_test_b = escalador_tabular.transform(
    SimpleImputer(strategy="median").fit(X_tab_b[idx_train_b]).transform(X_tab_b[idx_test_b])
)
X_img_train_b, X_img_test_b = X_img_b[idx_train_b], X_img_b[idx_test_b]
y_train_b, y_test_b = y_b[idx_train_b], y_b[idx_test_b]


def construir_modelo_fusion(input_shape_img=(TAMANO_IMG, TAMANO_IMG, 1), n_features_tab=9):
    # --- Rama de imagen (misma estructura que el Modelo A) ---
    entrada_img = layers.Input(shape=input_shape_img, name="entrada_imagen")
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(entrada_img)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    embedding_img = layers.Dense(64, activation="relu", name="embedding_imagen")(x)

    # --- Rama tabular ---
    entrada_tab = layers.Input(shape=(n_features_tab,), name="entrada_tabular")
    t = layers.Dense(32, activation="relu")(entrada_tab)
    t = layers.Dropout(0.2)(t)
    embedding_tab = layers.Dense(16, activation="relu", name="embedding_tabular")(t)

    # --- Fusión ---
    fusion = layers.Concatenate(name="fusion")([embedding_img, embedding_tab])
    f = layers.Dense(32, activation="relu")(fusion)
    f = layers.Dropout(0.4)(f)
    salida = layers.Dense(1, activation="sigmoid")(f)

    modelo = Model(inputs=[entrada_img, entrada_tab], outputs=salida, name="modelo_fusion_B")
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return modelo


modelo_fusion = construir_modelo_fusion(n_features_tab=X_tab_b.shape[1])
modelo_fusion.summary()


## **A2.5. Entrenamiento y evaluación del Modelo B**

In [ ]:
n_pos_b, n_neg_b = y_train_b.sum(), len(y_train_b) - y_train_b.sum()
peso_clases_b = {0: 1.0, 1: (n_neg_b / max(n_pos_b, 1))} if n_pos_b > 0 else {0: 1.0, 1: 1.0}

historial_b = modelo_fusion.fit(
    [X_img_train_b, X_tab_train_b], y_train_b,
    epochs=60, batch_size=8, validation_split=0.2,
    class_weight=peso_clases_b,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8,
                                                  restore_best_weights=True)],
    verbose=0,
)
print(f"Modelo B entrenado en {len(historial_b.history['loss'])} épocas.")

y_proba_b = modelo_fusion.predict([X_img_test_b, X_tab_test_b], verbose=0).ravel()
y_pred_b = (y_proba_b >= 0.5).astype(int)

print("\n=== Modelo B (CNN + tabular) — Evaluación a nivel de imagen ===")
print(classification_report(y_test_b, y_pred_b,
                             target_names=["Control (0)", "Epilepsia (1)"], zero_division=0))

auc_b = roc_auc_score(y_test_b, y_proba_b) if len(set(y_test_b)) > 1 else float("nan")
print(f"AUC-ROC: {auc_b:.3f}" if auc_b == auc_b else "AUC-ROC no definido (una sola clase en prueba).")

ConfusionMatrixDisplay(confusion_matrix(y_test_b, y_pred_b),
                        display_labels=["Control", "Epilepsia"]).plot(cmap="Greens", colorbar=False)
plt.title("Modelo B (fusión) — Matriz de confusión (nivel de imagen)")
plt.savefig(CARPETA_SALIDA + "modelo_b_matriz_confusion.png", dpi=140, bbox_inches="tight")
plt.show()

modelo_fusion.save(CARPETA_SALIDA + "modelo_b_fusion_cnn_tabular.keras")


# **Sección B — Benchmarking de 5 Clasificadores Clásicos sobre Métricas Tabulares**

## **B.1. Carga de features (reutiliza la salida del Archivo 1)**
En vez de volver a parsear los `*_tract_stat*.txt` desde cero, esta sección carga directamente
`features_por_sujeto.csv` — el artefacto que el Archivo 1 (Sección 5.2) ya exporta — para evitar
mantener dos implementaciones distintas del mismo parseo. Si ese archivo no existe todavía, se
genera el mismo dataset sintético correlacionado que usa el Archivo 1, para que esta sección también
sea ejecutable de forma independiente.

In [ ]:
def generar_features_sinteticas(n_sujetos=200, semilla=42):
    """Mismo generador sintético correlacionado que el Archivo 1 (Sección 2.4):
    señal conocida y controlada por grupo, nunca etiquetas desconectadas de los
    datos. Devuelve un DataFrame ya en formato ancho (una fila por sujeto).
    """
    rng = np.random.default_rng(semilla)
    tractos = ["arc", "unc", "mdlf", "ilf"]
    lados = ["izquierdo", "derecho"]
    filas = []
    for i in range(1, n_sujetos + 1):
        grupo = "control" if i <= n_sujetos // 2 else "epilepsia"
        fila = {"subject_id": f"S{i:03d}", "grupo_dx": grupo}
        for tracto in tractos:
            for lado in lados:
                media_fa = 0.45 if grupo == "control" else 0.38
                fila[f"{tracto}_{lado}_fa_mean"] = rng.normal(media_fa, 0.05)
                fila[f"{tracto}_{lado}_md_mean"] = rng.normal(
                    0.0008 if grupo == "control" else 0.0009, 0.0001)
                fila[f"{tracto}_{lado}_volume_mm3"] = rng.normal(1500, 200)
        filas.append(fila)
    return pd.DataFrame(filas)


if os.path.exists(RUTA_FEATURES_TABULARES):
    df_features = pd.read_csv(RUTA_FEATURES_TABULARES)
    print(f"'{RUTA_FEATURES_TABULARES}' cargado: {df_features.shape[0]} sujetos, "
          f"{df_features.shape[1] - 2} features.")
else:
    print(f"No se encontró '{RUTA_FEATURES_TABULARES}'; generando dataset SINTÉTICO de "
          f"validación (ver nota metodológica de la portada)...")
    df_features = generar_features_sinteticas()
    print(f"Dataset sintético generado: {df_features.shape[0]} sujetos, "
          f"{df_features.shape[1] - 2} features.")

df_features.head()


## **B.2. Benchmarking con selección recursiva de características (RFECV)**
A diferencia del Archivo 1 (que entrena los 5 modelos directamente sobre todas las features), aquí
se antepone `RFECV` (*Recursive Feature Elimination with Cross-Validation*) dentro del mismo
pipeline: elimina recursivamente las features menos relevantes, evaluando por validación cruzada,
antes de entrenar el clasificador final — útil cuando el número de features (hasta 4 tractos × 2
lados × ~10 métricas = 80) es grande respecto al número de sujetos. Como `RFECV` requiere un
estimador con `coef_` o `feature_importances_`, los modelos que no lo tienen (SVM, Gradient
Boosting) usan un `RandomForestClassifier` auxiliar solo para la selección de features, y luego
entrenan su propio clasificador final sobre las features seleccionadas.

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

y = df_features["grupo_dx"].map({"control": 0, "epilepsia": 1})
X = df_features.drop(columns=["subject_id", "grupo_dx"])

if y.nunique() < 2:
    print("Se requieren ambas clases ('control' y 'epilepsia') en 'grupo_dx' para benchmarking. "
          "Encontradas:", y.unique().tolist())
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    MODELOS_CANDIDATOS = {
        "Regresión Logística": LogisticRegression(max_iter=2000, class_weight="balanced",
                                                    random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=150, class_weight="balanced",
                                                  random_state=42),
        "SVM": SVC(probability=True, class_weight="balanced", random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "Multi-Layer Perceptron (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=800,
                                                         random_state=42),
    }

    resultados_benchmarking = []
    pipelines_entrenados = {}   # se reutilizan en la Sección B.3 (evaluación detallada)

    # RFECV necesita un estimador con 'coef_' (modelos lineales) o
    # 'feature_importances_' (modelos de árboles) -- ambos solo existen DESPUÉS
    # de entrenar, por lo que no se puede detectar con hasattr() de antemano;
    # se declara explícitamente qué modelos los tienen de forma nativa.
    MODELOS_CON_IMPORTANCIA_NATIVA = {"Regresión Logística", "Random Forest", "Gradient Boosting"}
    n_folds_rfecv = min(5, y_train.value_counts().min())

    print("=" * 85)
    print("ENTRENAMIENTO, RFECV Y BENCHMARKING DE 5 MODELOS")
    print("=" * 85)

    for nombre_modelo, estimador in MODELOS_CANDIDATOS.items():
        print(f"\nEntrenando: {nombre_modelo}...")

        estimador_selector = (estimador if nombre_modelo in MODELOS_CON_IMPORTANCIA_NATIVA
                               else RandomForestClassifier(random_state=42))
        selector_features = RFECV(
            estimator=estimador_selector, step=1, cv=StratifiedKFold(n_folds_rfecv),
            scoring="roc_auc", min_features_to_select=3,
        )

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("feature_selection", selector_features),
            ("clasificador", estimador),
        ])

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]

        resultados_benchmarking.append({
            "modelo": nombre_modelo,
            "accuracy": accuracy_score(y_test, y_pred),
            "auc_roc": roc_auc_score(y_test, y_proba) if y_test.nunique() > 1 else float("nan"),
            "n_features_seleccionadas": int(selector_features.n_features_),
        })
        pipelines_entrenados[nombre_modelo] = pipeline

    df_resultados_benchmarking = (
        pd.DataFrame(resultados_benchmarking).sort_values("auc_roc", ascending=False)
    )
    df_resultados_benchmarking.to_csv(
        CARPETA_SALIDA + "resultados_benchmarking_rfecv.csv", index=False, encoding="utf-8-sig"
    )

df_resultados_benchmarking if y.nunique() > 1 else None


## **B.3. Evaluación detallada por clase (verdaderos/falsos positivos y negativos)**
Reutiliza los **mismos pipelines ya entrenados** en B.2 (con imputación, escalado y selección de
features incluidos) — no se vuelve a entrenar un pipeline distinto que ignore esos pasos.

In [ ]:
if y.nunique() > 1:
    print("=" * 85)
    print("EVALUACIÓN DETALLADA DE CLASE (FALSOS POSITIVOS Y FALSOS NEGATIVOS)")
    print("=" * 85)

    for nombre_modelo, pipeline in pipelines_entrenados.items():
        y_pred = pipeline.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)
        tn, fp, fn, tp = cm.ravel()

        print(f"\nModelo: {nombre_modelo}")
        print(f"  Verdaderos Negativos (controles bien clasificados): {tn}")
        print(f"  Falsos Positivos (control catalogado como epilepsia): {fp}")
        print(f"  Falsos Negativos (epilepsia catalogado como control): {fn}")
        print(f"  Verdaderos Positivos (epilepsia bien detectada): {tp}")
        print(classification_report(y_test, y_pred,
                                     target_names=["Control (0)", "Epilepsia (1)"],
                                     zero_division=0))
        print("-" * 65)


# **Comparación Final — CNN vs. Machine Learning Clásico**

In [ ]:
filas_comparacion = []
if "auc_cnn" in globals():
    filas_comparacion.append({"enfoque": "Modelo A — CNN sobre TDI (nivel imagen)",
                               "accuracy": accuracy_score(y_test_img, y_pred_cnn),
                               "auc_roc": auc_cnn})
if "df_pred_sujeto" in globals():
    filas_comparacion.append({
        "enfoque": "Modelo A — CNN sobre TDI (nivel sujeto, voto por promedio)",
        "accuracy": accuracy_score(df_pred_sujeto["y_real"], df_pred_sujeto["y_pred"]),
        "auc_roc": (roc_auc_score(df_pred_sujeto["y_real"], df_pred_sujeto["y_proba_media"])
                    if df_pred_sujeto["y_real"].nunique() > 1 else float("nan")),
    })
if "auc_b" in globals():
    filas_comparacion.append({"enfoque": "Modelo B — Fusión CNN + tabular (nivel imagen)",
                               "accuracy": accuracy_score(y_test_b, y_pred_b),
                               "auc_roc": auc_b})
if "df_resultados_benchmarking" in globals() and y.nunique() > 1:
    mejor_clasico = df_resultados_benchmarking.iloc[0]
    filas_comparacion.append({"enfoque": f"Mejor clásico: {mejor_clasico['modelo']}",
                               "accuracy": mejor_clasico["accuracy"],
                               "auc_roc": mejor_clasico["auc_roc"]})

df_comparacion_final = pd.DataFrame(filas_comparacion)
df_comparacion_final.to_csv(CARPETA_SALIDA + "comparacion_cnn_vs_clasico.csv", index=False,
                             encoding="utf-8-sig")
df_comparacion_final.round(3)


# **Despliegue — Persistencia de Artefactos**

In [ ]:
modelo_cnn.save(CARPETA_SALIDA + "modelo_cnn_tdi.keras")

referencias_md = """# Referencias Metodológicas — Archivo 3 (Modelado)

## Arquitectura CNN y buenas prácticas de regularización
Ioffe, S., Szegedy, C. (2015). Batch normalization: Accelerating deep network training by
reducing internal covariate shift. ICML.

Srivastava, N. et al. (2014). Dropout: A simple way to prevent neural networks from
overfitting. Journal of Machine Learning Research, 15(1), 1929-1958.

## Selección de características (RFECV)
Guyon, I., Weston, J., Barnhill, S., Vapnik, V. (2002). Gene selection for cancer
classification using support vector machines. Machine Learning, 46, 389-422.

## Validación agrupada por sujeto (prevención de fuga de datos)
Ver Archivo 1, Sección 5.1 -- mismo principio aplicado aquí a nivel de imagen
(GroupShuffleSplit por subject_id).

## Track Density Imaging (TDI) como entrada de visión por computador
Calamante, F., Tournier, J-D., Jackson, G.D., Connelly, A. (2010). Track-density imaging
(TDI): Super-resolution white matter imaging using whole-brain track-density mapping.
NeuroImage, 53(4), 1233-1243. https://doi.org/10.1016/j.neuroimage.2010.07.024
"""
with open(CARPETA_SALIDA + "referencias_archivo3.md", "w", encoding="utf-8") as f:
    f.write(referencias_md)

print(f"Artefactos guardados en {CARPETA_SALIDA}:")
print(" - modelo_cnn_tdi.keras                      (Modelo A, Sección A.5-A.6)")
print(" - modelo_b_fusion_cnn_tabular.keras          (Modelo B, Sección A2.4-A2.5)")
print(" - cnn_curvas_entrenamiento.png, cnn_matriz_confusion.png   (Modelo A, Sección A.6)")
print(" - modelo_b_matriz_confusion.png              (Modelo B, Sección A2.5)")
print(" - resultados_benchmarking_rfecv.csv          (Sección B.2)")
print(" - comparacion_cnn_vs_clasico.csv             (Comparación final: A, B y mejor clásico)")
print(" - referencias_archivo3.md")


# **Conclusión**
- Se corrigieron dos errores reales de una versión previa de este notebook: (1) la variable
  `modelos` referenciada en el bloque de evaluación detallada nunca estaba definida (usaba
  `MODELOS_CANDIDATOS`) — causaba `NameError`; (2) el bucle principal de benchmarking calculaba
  `accuracy`/`auc_roc` por modelo pero nunca los agregaba a `resultados_benchmarking`, que quedaba
  vacío — ambos corregidos en la Sección B.2-B.3, que ahora reutiliza los mismos pipelines ya
  entrenados (con imputación y escalado) para la evaluación detallada, en vez de reentrenar un
  pipeline distinto e inconsistente.
- Se eliminaron los bloques que entrenaban con **etiquetas generadas aleatoriamente** — cualquier
  resultado de esa versión no tiene validez clínica, sin importar el *accuracy* reportado.
- Se construyó un pipeline de **Modelo A — CNN sobre imágenes TDI reales** (Sección A), con
  partición agrupada por sujeto (igual principio anti-fuga que el Archivo 1) y agregación a nivel de
  paciente por voto de probabilidad promedio.
- Se agregó el **Modelo B — fusión multimodal** (Sección A2): una segunda red, de dos ramas, que
  combina la misma imagen TDI con las métricas reales de difusión (FA/MD/AD/RD) que exporta el
  Archivo 2 (Sección 3.5), emparejadas 1 a 1 por `(subject_id, cluster_tag)` — para cuantificar si
  la microestructura aporta señal que la forma del tracto por sí sola no captura.
- El **benchmarking de 5 clasificadores clásicos** (Sección B) reutiliza directamente
  `features_por_sujeto.csv` del Archivo 1 en vez de duplicar su lógica de parseo, añadiendo RFECV
  como paso de selección de características.
- La tabla de **Comparación Final** ahora contrasta los tres enfoques (Modelo A, Modelo B y el
  mejor clasificador clásico) bajo las mismas métricas (*accuracy*, AUC-ROC).
- **Siguiente paso recomendado:** correr el Archivo 2 para cada sujeto real de la cohorte (genera,
  por sujeto, tanto las imágenes TDI como `metricas_reales_por_tracto.csv` bajo
  `<subject_id>_<grupo_dx>/`), descomprimir todas esas carpetas dentro de `tdi_cohorte/`, y correr
  el Archivo 1 sobre el conjunto completo de `*_tract_stat*.txt` — este notebook detecta
  automáticamente esos datos reales y deja de usar los conjuntos sintéticos de validación.
